# TEMPLATE UTS DM II — CLUSTERING AUDIO (Google Colab)

**Cara tercepat:** jalankan `SUMBER="contoh"` untuk melihat alurnya. Pada UTS ubah jadi `"upload_zip"` (ZIP banyak WAV/MP3) atau `"upload_audio"` (satu WAV/MP3).

**Dua unit clustering:** `UNIT="file"` = setiap file audio menjadi **1 observasi**. `UNIT="segmen"` = setiap potongan berdurasi tetap menjadi **1 observasi**, berguna jika hanya 1 file panjang.

**Alur:** load → cek durasi/sample rate/channel/bit depth (jika WAV PCM) → waveform → spectrogram amplitudo + dB → MFCC/rms/ZCR → sampling → StandardScaler → Elbow/Silhouette → K-Means → PCA → interpretasi.

Data contoh dihasilkan sebagai **suara sintetis WAV lokal** (tanpa perlu download internet selain instal library), bukan data ucapan manusia.

**Keyword:** `librosa.load sr none mono`, `soundfile.info WAV channels`, `librosa stft specshow`, `librosa amplitude_to_db`, `librosa feature mfcc`, `KMeans audio clustering`.

## 0. KONFIGURASI — ganti hanya bagian ini

In [ ]:
SUMBER = "contoh"         # "contoh", "upload_zip", "upload_audio", "folder"
FOLDER_DATA = "/content/drive/MyDrive/dataset_audio"  # khusus SUMBER="folder"
UNIT = "file"             # "file" satu data/file; "segmen" satu data/potongan waktu
N_SAMPEL_FILE = 100      # sample maksimal banyak file, misalnya 100 atau 300
N_SAMPEL_SEGMEN = 300    # sample maksimal potongan audio setelah ekstraksi
DURASI_BACA = 5          # maksimal detik per file agar proses cepat
DURASI_SEGMEN = 1.5      # untuk UNIT="segmen" dalam detik
HOP_SEGMEN = 1.0         # geser antarsegmen dalam detik
SR_FITUR = 16000        # sample rate standar ekstraksi MFCC (resampling)
K_FINAL = 3
RANDOM_STATE = 42
print("Sumber:", SUMBER, "| Unit:", UNIT, "| Maks file:", N_SAMPEL_FILE, "| k:", K_FINAL)

**Sesudah menjalankan cell — petunjuk & interpretasi:** **Pilih UNIT:** bila soal menyediakan banyak rekaman suara, gunakan `"file"`. Bila hanya satu rekaman panjang dan harus clustering, `"segmen"`. Sampling file dipakai **sebelum ekstraksi** agar tidak berat. **Keyword:** `sample audio files random choice`, `audio segmentation window hop`.

## 1. Instalasi dan import library

In [ ]:
%pip -q install librosa soundfile
import os
from pathlib import Path
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import librosa.display
import soundfile as sf
from IPython.display import display, Audio
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

print("Library utama: librosa (audio & fitur), soundfile (metadata dan WAV)")
print("Pandas: pd.read_csv(...) bila tersedia tabel metadata, bukan gelombang mentah")
print("librosa.load(...) untuk gelombang; sf.info(...) untuk metadata format terdukung")

**Sesudah menjalankan cell — petunjuk & interpretasi:** `librosa` membaca waveform/menghitung spectrogram/MFCC. `soundfile` membaca metadata WAV dan menulis WAV. `numpy` array suara; `sklearn` clustering. **Keyword:** `import librosa.display`, `soundfile info subtype PCM_16`.

## 2. Baca atau upload file; sampling daftar berkas

In [ ]:
if SUMBER == "contoh":
    root = Path("/content/uts_audio_contoh")
    root.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(RANDOM_STATE)
    sr_contoh = 16000
    t = np.arange(int(2.8 * sr_contoh)) / sr_contoh
    for grup, freq in enumerate([220, 440, 880, 1760]):
        for no in range(4):
            sinyal = 0.3*np.sin(2*np.pi*(freq + no*8)*t) + rng.normal(0, 0.01, len(t))
            sf.write(root / f"grup{grup}_contoh{no}.wav", sinyal, sr_contoh, subtype="PCM_16")
    print("16 WAV sintetis telah dibuat.")
elif SUMBER in ("upload_zip", "upload_audio"):
    from google.colab import files
    uploaded = files.upload()
    nama_file = next(iter(uploaded))
    if SUMBER == "upload_zip":
        root = Path("/content/uts_audio_upload")
        root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(nama_file) as z:
            z.extractall(root)
    else:
        root = Path(".")
        print("Audio terunggah:", nama_file)
else:
    root = Path(FOLDER_DATA)

if SUMBER == "upload_audio":
    paths = [Path(nama_file)]
else:
    paths = sorted([p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in (".wav", ".flac", ".mp3", ".ogg")])
assert len(paths) > 0, "File audio tidak ditemukan. Pilih WAV/FLAC/MP3/OGG atau cek folder."
n_total = len(paths)
idx = np.random.default_rng(RANDOM_STATE).choice(n_total, min(N_SAMPEL_FILE, n_total), replace=False)
paths = [paths[j] for j in idx]
print("Jumlah file tersedia:", n_total, "| dipilih (sampling):", len(paths))
display(pd.DataFrame({"file": [p.name for p in paths]}).head(8))

**Sesudah menjalankan cell — petunjuk & interpretasi:** **Struktur ZIP bebas** (subfolder diperbolehkan). `N_SAMPEL_FILE = 100` membuat maksimal 100 file diproses meskipun ada 1000 file. Jika hanya punya satu rekaman, gunakan `UNIT="segmen"` agar jumlah sampel lebih dari satu. **Keyword:** `colab upload zip audio dataset`, `pathlib rglob wav files`.

## 3. Metadata audio: sample rate, channel, durasi, subtype dan bitrate

In [ ]:
metadata = []
for p in paths[:min(10, len(paths))]:
    # Pertahankan channel dan sample rate sumber asli
    y_asli, sr_asli = librosa.load(str(p), sr=None, mono=False, duration=DURASI_BACA)
    channel = 1 if y_asli.ndim == 1 else y_asli.shape[0]
    durasi_loaded = y_asli.shape[-1]/sr_asli
    if p.suffix.lower() == ".wav":
        info = sf.info(str(p))
        tipe = info.subtype  # contoh PCM_16, PCM_24
        bits = int(tipe.split("_")[1]) if tipe.startswith("PCM_") and tipe.split("_")[1].isdigit() else np.nan
        bitrate_kbps = sr_asli*channel*bits/1000 if not np.isnan(bits) else np.nan
    else:
        tipe, bits, bitrate_kbps = "terkompresi/lainnya", np.nan, np.nan
    metadata.append({"file":p.name, "sample_rate_Hz":sr_asli, "channel":channel,
                     "durasi_dibaca_detik":round(durasi_loaded,2), "subtype":tipe,
                     "bit_depth_PCM":bits, "bitrate_PCM_kbps":bitrate_kbps,
                     "ukuran_file_kB":round(p.stat().st_size/1024,1)})
df_meta = pd.DataFrame(metadata)
display(df_meta)

**Sesudah menjalankan cell — petunjuk & interpretasi:** **Sample rate** = sampel suara/detik (Hz), **channel** = 1 mono/2 stereo, **bit depth** = bit tiap sampel PCM, **bitrate PCM tidak terkompresi** ≈ `sample_rate × channel × bit_depth` bit/detik. Untuk MP3/OGG, bitrate **terkompresi** berbeda dan tidak diestimasi dari rumus PCM; tabel berisi NaN. `durasi_dibaca_detik` dibatasi `DURASI_BACA`, bukan selalu durasi total berkas. **Keyword:** `sample rate vs bit depth vs bitrate audio`, `soundfile.info subtype`.

## 4. Waveform (amplitudo vs waktu) + dengarkan contoh

In [ ]:
y_tampil, sr_tampil = librosa.load(str(paths[0]), sr=None, mono=True, duration=DURASI_BACA)
print("File:", paths[0].name, "| shape waveform:", y_tampil.shape, "| sr:", sr_tampil)
print("Durasi dibaca:", round(len(y_tampil)/sr_tampil,2), "detik")
plt.figure(figsize=(11, 3))
librosa.display.waveshow(y_tampil, sr=sr_tampil)
plt.title("Waveform (amplitudo vs waktu)"); plt.xlabel("Waktu (detik)")
plt.tight_layout(); plt.show()
display(Audio(y_tampil, rate=sr_tampil))

**Sesudah menjalankan cell — petunjuk & interpretasi:** Waveform memperlihatkan amplitudo sinyal terhadap waktu; amplitudo numerik bukan otomatis dB. Peak/bagian sunyi terlihat melalui tinggi-rendahnya gelombang. **Keyword:** `librosa.display.waveshow`.

## 5. Spectrogram MAGNITUDO/AMPLITUDO

In [ ]:
S = np.abs(librosa.stft(y_tampil))
print("Shape STFT magnitudo (frequency bins, time frames):", S.shape)
plt.figure(figsize=(10, 4))
librosa.display.specshow(S, sr=sr_tampil, x_axis="time", y_axis="linear")
plt.colorbar(label="Magnitudo")
plt.title("Spectrogram magnitudo (linear)"); plt.tight_layout(); plt.show()

**Sesudah menjalankan cell — petunjuk & interpretasi:** `STFT` membagi suara menjadi potongan waktu dan mengukur kontribusi frekuensi. Warna menunjukkan **magnitudo spektrum**, bukan besaran waveform dalam volt. **Keyword:** `librosa stft magnitude spectrogram specshow`.

## 6. Spectrogram DESIBEL (dB)

In [ ]:
S_db = librosa.amplitude_to_db(S, ref=np.max)
print("Shape spectrogram dB:", S_db.shape, "| rentang:", round(float(S_db.min()),1), "s.d.", round(float(S_db.max()),1))
plt.figure(figsize=(10, 4))
librosa.display.specshow(S_db, sr=sr_tampil, x_axis="time", y_axis="log", cmap="magma")
plt.colorbar(label="dB relatif puncak")
plt.title("Spectrogram dB (relatif magnitudo maksimum)"); plt.tight_layout(); plt.show()

**Sesudah menjalankan cell — petunjuk & interpretasi:** Konversi `librosa.amplitude_to_db(S, ref=np.max)` menghasilkan desibel **relatif terhadap magnitudo maksimum (0 dB)**. Ini **bukan** pengukuran kebisingan absolut dB SPL dari mikrofon. **Keyword:** `librosa amplitude_to_db ref np.max`, `log frequency spectrogram`.

## 7. Ekstraksi fitur MFCC + RMS + ZCR; jika perlu potong jadi segmen

In [ ]:
# Data clustering = vektor angka, bukan waveform mentah
fitur = []
identitas = []
for p in paths:
    y, sr = librosa.load(str(p), sr=SR_FITUR, mono=True, duration=DURASI_BACA)
    if UNIT == "file":
        bagian = [y]
    else:
        lebar = int(DURASI_SEGMEN * sr)
        langkah = int(HOP_SEGMEN * sr)
        bagian = [y[s:s+lebar] for s in range(0, len(y), langkah) if len(y[s:s+lebar]) >= int(0.75*sr)]
    for j, seg in enumerate(bagian):
        mfcc = librosa.feature.mfcc(y=seg, sr=sr, n_mfcc=13)
        rms = librosa.feature.rms(y=seg).mean()
        zcr = librosa.feature.zero_crossing_rate(y=seg).mean()
        centroid = librosa.feature.spectral_centroid(y=seg, sr=sr).mean()
        fitur.append(list(mfcc.mean(axis=1)) + [rms, zcr, centroid])
        identitas.append({"file":p.name, "segmen":j})
X_raw = np.array(fitur)
df_id = pd.DataFrame(identitas)
print("Shape fitur (observasi, 13 MFCC + RMS + ZCR + centroid):", X_raw.shape)
display(pd.DataFrame(X_raw[:5], columns=[f"MFCC_{j}" for j in range(1,14)]+["RMS","ZCR","Centroid"]).round(3))

**Sesudah menjalankan cell — petunjuk & interpretasi:** **MFCC** meringkas karakter spektrum suara; **RMS** energi amplitudo; **ZCR** frekuensi perpindahan tanda; **spectral centroid** rerata frekuensi berbobot. Unit `file` memberi satu baris per audio; unit `segmen` memberi banyak baris per audio. `DURASI_BACA` membatasi audio yang dianalisis, naikkan jika ingin segmen dari file panjang. **Keyword:** `librosa feature mfcc mean axis 1`, `zero_crossing_rate`, `spectral_centroid`.

## 8. Sampling observasi/segmen dan StandardScaler

In [ ]:
assert len(X_raw) >= 3, "Data clustering minimal 3 observasi. Jika hanya satu audio, gunakan UNIT='segmen' atau tambah file."
if UNIT == "segmen":
    idx_seg = np.random.default_rng(RANDOM_STATE).choice(len(X_raw), min(N_SAMPEL_SEGMEN,len(X_raw)), replace=False)
    X_raw = X_raw[idx_seg]
    df_id = df_id.iloc[idx_seg].reset_index(drop=True)
    print("Sampling segmen dilakukan:", len(X_raw), "baris")
else:
    print("Sampel file sudah dibatasi sebelum ekstraksi:", len(X_raw), "baris")
X = StandardScaler().fit_transform(X_raw)
print("Shape sebelum scaling:", X_raw.shape, "| setelah scaling:", X.shape)
display(df_id.head())

**Sesudah menjalankan cell — petunjuk & interpretasi:** `StandardScaler` membantu menghindari MFCC, RMS, dan centroid saling mendominasi hanya karena beda skala. Untuk ribuan segmen, pilih `N_SAMPEL_SEGMEN=100`–`1000`. **Keyword:** `sklearn StandardScaler audio features`, `numpy random choice without replacement`.

## 9. Elbow & Silhouette untuk pilihan k

In [ ]:
n_unik = len(np.unique(X, axis=0))
assert n_unik >= 3, "Fitur audio terlalu seragam; gunakan audio lebih bervariasi."
k_candidates = list(range(2, min(7, len(X)-1, n_unik)+1))
inertia, sil = [], []
for k in k_candidates:
    m = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    lab = m.fit_predict(X)
    inertia.append(m.inertia_)
    sil.append(silhouette_score(X, lab, sample_size=min(300, len(X)), random_state=RANDOM_STATE) if len(set(lab))>1 else np.nan)
display(pd.DataFrame({"k":k_candidates, "inertia":inertia, "silhouette":sil}).round(3))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(k_candidates, inertia, "o-"); axes[0].set(title="Elbow", xlabel="k", ylabel="Inertia")
axes[1].plot(k_candidates, sil, "o-"); axes[1].set(title="Silhouette", xlabel="k", ylabel="Skor")
plt.tight_layout(); plt.show()

**Sesudah menjalankan cell — petunjuk & interpretasi:** Silhouette mendekati +1 cenderung baik, mendekati 0 menunjukkan cluster bersinggungan, nilai negatif menunjukkan kemungkinan salah penempatan. Gunakan `K_FINAL` sesuai grafik dan isi soal. **Keyword:** `Elbow Silhouette KMeans MFCC`.

## 10. K-Means final dan evaluasi

In [ ]:
K = min(K_FINAL, len(X)-1, n_unik)
model = KMeans(n_clusters=K, random_state=RANDOM_STATE, n_init=10)
cluster = model.fit_predict(X)
df_hasil = df_id.copy()
df_hasil["cluster"] = cluster
print("K =", K)
display(df_hasil["cluster"].value_counts().sort_index().to_frame("jumlah"))
if 1 < len(set(cluster)) < len(X):
    print("Silhouette:", round(silhouette_score(X, cluster),4))
    print("Davies-Bouldin:", round(davies_bouldin_score(X, cluster),4))
else:
    print("Evaluasi tidak terdefinisi untuk 1 cluster saja.")

**Sesudah menjalankan cell — petunjuk & interpretasi:** K-Means mengelompokkan **karakteristik akustik**, bukan otomatis mengenali kata, penyanyi, emosi, atau jenis musik. Interpretasi perlu mendengarkan/meneliti contoh audio tiap cluster. **Keyword:** `sklearn cluster KMeans audio MFCC`, `davies_bouldin_score`.

## 11. PCA 2D, statistik cluster dan contoh file

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
coords = pca.fit_transform(X)
print("Shape PCA untuk visualisasi:", coords.shape)
plt.figure(figsize=(7,5)); plt.scatter(coords[:,0],coords[:,1],c=cluster,cmap="tab10",s=30)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("Clustering fitur audio (PCA)")
plt.tight_layout(); plt.show()
print("Contoh file/segmen per cluster:")
for c in range(K):
    print("Cluster",c,":",df_hasil[df_hasil.cluster==c]["file"].head(5).tolist())
print("Rata-rata fitur per cluster (MFCC pertama, RMS, ZCR, centroid):")
df_feat = pd.DataFrame(X_raw,columns=[f"MFCC_{j}" for j in range(1,14)]+["RMS","ZCR","Centroid"])
df_feat["cluster"] = cluster
display(df_feat.groupby("cluster")[["MFCC_1","RMS","ZCR","Centroid"]].mean().round(3))

**Sesudah menjalankan cell — petunjuk & interpretasi:** PCA hanya untuk tampilan 2D; hasil cluster dihitung pada semua fitur yang sudah diskalakan. Mean MFCC/RMS/ZCR membantu mendeskripsikan profil suara, tetapi tidak bisa disamakan langsung dengan label semantik. **Keyword:** `PCA audio features scatter`, `pandas groupby mean cluster features`.

## 12. Simpan CSV hasil

In [ ]:
df_hasil.to_csv("hasil_cluster_audio.csv", index=False)
print("CSV tersimpan: hasil_cluster_audio.csv")
display(df_hasil.head(10))

**Sesudah menjalankan cell — petunjuk & interpretasi:** Simpan atau unduh CSV dari panel Files Colab; hasil memuat nama file, indeks segmen, dan cluster. **Keyword:** `pandas dataframe to_csv`, `colab download`.

## Checklist UTS — Audio

1. Upload WAV/MP3 atau ZIP. 2. Pastikan **unit analisis** = file atau segmen. 3. Periksa sample rate, channel, durasi; bit depth/bitrate PCM hanya untuk WAV PCM yang sesuai. 4. Plot waveform. 5. STFT magnitudo & dB relatif puncak. 6. MFCC (rata-rata 13 fitur) + RMS + ZCR + spectral centroid. 7. Sampling. 8. StandardScaler. 9. Elbow/Silhouette. 10. K-Means + evaluasi. 11. PCA 2D + interpretasi.

**Contoh:** jika punya 5000 rekaman, `N_SAMPEL_FILE=100` mengambil 100 **file** secara acak sebelum ekstraksi; jika satu rekaman 60 detik, `UNIT="segmen"`, `DURASI_BACA=60`, `DURASI_SEGMEN=1.5` membentuk banyak baris fitur (sebelum sampling).